v IMPORT DATA v

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("Downloads/archive (1)/college_major_roi.csv")

print(df.shape)

(30000, 18)


In [2]:
df.info()
df.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   grad_id                         30000 non-null  object 
 1   major                           30000 non-null  object 
 2   major_category                  30000 non-null  object 
 3   institution_tier                30000 non-null  object 
 4   institution_selectivity_pctile  30000 non-null  int64  
 5   gpa                             30000 non-null  float64
 6   had_internship                  30000 non-null  int64  
 7   completed_on_time               30000 non-null  int64  
 8   region                          30000 non-null  object 
 9   net_cost_usd                    30000 non-null  int64  
 10  debt_usd                        30000 non-null  int64  
 11  hs_baseline_10yr_usd            30000 non-null  int64  
 12  added_earnings_10yr_usd         

,grad_id,major,major_category,institution_tier,institution_selectivity_pctile,gpa,had_internship,completed_on_time,region,net_cost_usd,debt_usd,hs_baseline_10yr_usd,added_earnings_10yr_usd,earnings_10yr_usd,net_roi_usd,roi_pct,positive_roi,high_roi
0,G000000,biology,STEM,mid_tier,52,2.58,0,0,south,144400,0,411700,101000,512700,-43400,-30.0,0,0
1,G000001,marketing,business,mid_tier,61,2.58,0,1,northeast,57400,0,356900,330600,687500,273200,476.1,1,0
2,G000002,electrical_engineering,STEM,top_public,78,3.14,1,1,northeast,65100,39600,339600,655100,994700,590000,906.4,1,1
3,G000003,marketing,business,top_public,77,3.04,0,1,midwest,105800,81300,399600,197500,597100,91700,86.8,1,0
4,G000004,marketing,business,ivy_elite,100,3.81,0,0,south,219200,93500,335700,222900,558600,3800,1.7,1,0
5,G000005,business,business,regional,37,3.91,1,0,northeast,79000,0,413800,466800,880600,387900,491.2,1,0
6,G000006,economics,business,regional,44,2.32,1,1,midwest,39500,0,388800,405200,794100,365800,926.3,1,0
7,G000007,social_work,social_science,mid_tier,59,4.00,1,1,midwest,59300,43500,350300,162400,512600,103100,174.0,1,0
8,G000008,mechanical_engineering,STEM,regional,45,3.61,0,1,northeast,62200,0,314300,344500,658800,282300,454.2,1,0
9,G000009,finance,business,top_public,79,2.46,1,0,south,109000,107500,307200,438000,745200,329000,301.8,1,0


v DATA CLEANING (MISSING VALUE IMPUTATION) v

In [5]:
df.isnull().sum()

grad_id                           0
major                             0
major_category                    0
institution_tier                  0
institution_selectivity_pctile    0
gpa                               0
had_internship                    0
completed_on_time                 0
region                            0
net_cost_usd                      0
debt_usd                          0
hs_baseline_10yr_usd              0
added_earnings_10yr_usd           0
earnings_10yr_usd                 0
net_roi_usd                       0
roi_pct                           0
positive_roi                      0
high_roi                          0
dtype: int64

v DATA CLEANING (DUPLICATE REMOVAL) v

In [24]:
print("Duplicated rows:", (df.duplicated().sum()))

Duplicated rows: 0


v DATA CLEANING (NOISE DREUCTION) v

In [10]:
# Check possible invalid values
print("Invalid GPA:", (df['gpa'] < 0).sum() + (df['gpa'] > 4).sum())

print("Invalid debt:", (df['debt_usd'] < 0).sum())

print("Invalid net cost:", (df['net_cost_usd'] < 0).sum())

Invalid GPA: 0
Invalid debt: 0
Invalid net cost: 0


v FEATURE ENGINEERING (FEATURE SELECTION) v

In [15]:
target = "high_roi"

remove_cols = [
    "grad_id",
    "net_roi_usd",
    "roi_pct",
    "earnings_10yr_usd",
    "added_earnings_10yr_usd",
    "positive_roi"
]

X = df.drop(columns=[target] + remove_cols)
y = df[target]

print("Features:")
print(X.columns.tolist())

Features:
['major', 'major_category', 'institution_tier', 'institution_selectivity_pctile', 'gpa', 'had_internship', 'completed_on_time', 'region', 'net_cost_usd', 'debt_usd', 'hs_baseline_10yr_usd']


v DATA SPLITTING v

In [22]:
from sklearn.model_selection import train_test_split

# 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Split the remaining 30% into 15% validation and 15% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (21000, 11)
Validation: (4500, 11)
Testing: (4500, 11)


v DATA TRANSFORMATION (STANDARDISATION) + FEATURE ENGINEERING (ONE HOT ENCODING) v

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_cols = [
    "institution_selectivity_pctile",
    "gpa",
    "had_internship",
    "completed_on_time",
    "net_cost_usd",
    "debt_usd",
    "hs_baseline_10yr_usd"
]

categorical_cols = [
    "major",
    "major_category",
    "institution_tier",
    "region"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", StandardScaler(), numerical_cols),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed.")
print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Testing:", X_test_processed.shape)

Preprocessing completed.
Training: (21000, 42)
Validation: (4500, 42)
Testing: (4500, 42)


v DATA BALANCING CHECK v

In [26]:
print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True))


Training class distribution:
high_roi
0    15750
1     5250
Name: count, dtype: int64

Training class proportions:
high_roi
0    0.75
1    0.25
Name: proportion, dtype: float64
